In [2]:
# -*- coding: utf-8 -*-
"""
RQ2 filter (core only) + aggregated gmd_present + outputs (main + URL list)

Core RQ2 (per repo, default-branch row in main dataset):
  Instru_test == 1
  AND any(test_invocation) ∈ {"Gradle","ADB"}
  AND any(execution_environment) ∈ {"Emulator","GMD","GMD_Intent"}

NOTE: We still aggregate gmd_present=True/False per repo from the config CSV
      and add it to the main dataset, but it is NOT used in the RQ2 condition.

Inputs (under ROOT):
  - Main dataset : 5.0_Total_Repo.csv
  - Config scan  : 3.1.1_Instru_T_Signal_ConfigV6.0.csv

Outputs:
  - URL_List_RQ2.csv          (repo_url, full_name)
  - 5.0_Total_Repo_2.csv      (main dataset + gmd_present, RQ2)
"""

from pathlib import Path
import pandas as pd
import re

# ---------- paths ----------
ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet")
MAIN_CSV      = ROOT / "5.0_Total_Repo.csv"
CONFIG_CSV    = ROOT / "3.1.1_Instru_T_Signal_ConfigV6.0.csv"
OUT_URLS      = ROOT / "URL_List_RQ2.csv"
MAIN_OUT_CSV2 = ROOT / "5.0_Total_Repo.csv"

# ---------- helpers ----------
def _to01(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return (pd.to_numeric(series, errors="coerce").fillna(0) > 0).astype(int)
    s = series.astype(str).str.strip().str.lower()
    token_map = {
        "1":1,"true":1,"yes":1,"y":1,"t":1,"on":1,
        "0":0,"false":0,"no":0,"n":0,"f":0,"off":0,"":0,"none":0,"null":0,"na":0,"nan":0
    }
    mapped = s.map(token_map)
    num = pd.to_numeric(s.str.replace(r"[^0-9\.\-]+", "", regex=True), errors="coerce")
    num01 = (num.fillna(0) > 0).astype(int)
    return mapped.where(mapped.notna(), num01).astype(int)

def _first_present(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols:
            return cols[c.lower()]
    return None

def _derive_full_name(df: pd.DataFrame) -> pd.Series:
    # prefer explicit 'full_name'
    full_col = _first_present(df, ["full_name","fullName","repo_full_name","owner/repo","owner_repo"])
    if full_col:
        out = df[full_col].astype(str).str.strip().str.lower()
        out = out.str.replace(r"__", "/", regex=True)
        out = out.str.replace(r"^([^/]+)\.([^/]+)$", r"\1/\2", regex=True)
        return out
    # else try owner + repo
    owner_col = _first_present(df, ["owner","org","login"])
    repo_col  = _first_present(df, ["repo","name","repository"])
    if owner_col and repo_col:
        return (df[owner_col].astype(str).str.strip().str.lower()
                + "/" + df[repo_col].astype(str).str.strip().str.lower())
    # else try a URL column
    url_col = _first_present(df, ["repo_url","url","html_url","clone_url","git_url","ssh_url","remote","homepage"])
    if url_col:
        rx = re.compile(r"github\.com/([^/\s]+)/([^/\s\.#]+)", re.I)
        def pick(u: str) -> str | None:
            m = rx.search(u or "")
            return (m.group(1) + "/" + m.group(2)).lower() if m else None
        return df[url_col].astype(str).map(pick)
    return pd.Series([None]*len(df), index=df.index, dtype="object")

def _derive_repo_url(df: pd.DataFrame, full_name: pd.Series) -> pd.Series:
    url_col = _first_present(df, ["repo_url","url","html_url","clone_url","git_url","ssh_url","remote"])
    if url_col:
        urls = df[url_col].astype(str).str.strip()
        needs_https = ~urls.str.contains(r"^https?://", na=True, regex=True)
        fallback = "https://github.com/" + full_name.fillna("")
        return urls.where(~needs_https, fallback)
    return "https://github.com/" + full_name.fillna("")

SEP_RE = re.compile(r"[,\s;/|/]+")

def _contains_any(cell: object, choices: set[str]) -> bool:
    s = str(cell or "").strip().lower()
    if not s:
        return False
    toks = [t for t in SEP_RE.split(s) if t]
    return any(t in choices for t in toks)

# ---------- load ----------
df_main = pd.read_csv(MAIN_CSV, encoding="utf-8-sig")
df_cfg  = pd.read_csv(CONFIG_CSV, encoding="utf-8-sig")

# ---------- normalize join key ----------
df_main["__full__"] = _derive_full_name(df_main)
df_cfg["__full__"]  = _derive_full_name(df_cfg)

df_main = df_main.dropna(subset=["__full__"]).copy()
df_cfg  = df_cfg.dropna(subset=["__full__"]).copy()

# ---------- identify columns in main ----------
col_instru = _first_present(df_main, ["Instru_test","instru_test","at","androidtest_present"])
col_invoc  = _first_present(df_main, ["test_invocation"])
col_env    = _first_present(df_main, ["execution_environment"])
main_full_col = _first_present(df_main, ["full_name","fullName","repo_full_name","owner/repo","owner_repo"])

missing = [name for name, col in {
    "Instru_test": col_instru,
    "test_invocation": col_invoc,
    "execution_environment": col_env,
}.items() if col is None]
if missing:
    raise KeyError(f"Missing required column(s) in main dataset: {missing}")

# ---------- aggregate gmd_present per repo from config ----------
col_gmd_cfg = _first_present(df_cfg, ["gmd_present","GMD_present","gmd","gmd_signal"])
if col_gmd_cfg is None:
    raise KeyError("Missing 'gmd_present' (or alias) in config CSV.")

df_cfg["__gmd_bool__"] = _to01(df_cfg[col_gmd_cfg]).astype(bool)
gmd_any_by_full = df_cfg.groupby("__full__")["__gmd_bool__"].any().astype(bool)

# ---- add aggregated gmd_present (True/False) to MAIN ----
df_main["gmd_present"] = df_main["__full__"].map(gmd_any_by_full).fillna(False).astype(bool)

# ---------- build RQ2 mask (CORE ONLY; no gmd_present fallback) ----------
invoc_ok = {"gradle","adb"}
env_ok   = {"emulator","gmd","gmd_intent"}

p_instru = _to01(df_main[col_instru]).eq(1)
p_invoc  = df_main[col_invoc].apply(_contains_any, args=(invoc_ok,))
p_env    = df_main[col_env].apply(_contains_any, args=(env_ok,))

mask_rq2 = (p_instru & p_invoc & p_env)

# ---- add RQ2 (True/False) ----
df_main["RQ2"] = mask_rq2.astype(bool)

# ---------- write URL list (repos that matched) ----------
picked = df_main.loc[df_main["RQ2"], ["__full__"]].drop_duplicates().copy()
if picked.empty:
    out_urls = pd.DataFrame(columns=["repo_url","full_name"])
else:
    picked = picked.merge(df_main, on="__full__", how="left")
    picked["repo_url"] = _derive_repo_url(picked, picked["__full__"])
    if main_full_col:
        full_out = picked[main_full_col].astype(str)
    else:
        full_out = picked["__full__"]
    out_urls = pd.DataFrame({
        "repo_url": picked["repo_url"],
        "full_name": full_out
    }).drop_duplicates()

# ---------- save outputs ----------
ROOT.mkdir(parents=True, exist_ok=True)
out_urls.to_csv(OUT_URLS, index=False, encoding="utf-8-sig")

# drop temp join col before saving the updated main (to _2)
df_main_out = df_main.copy()
if "__full__" in df_main_out.columns:
    df_main_out.drop(columns=["__full__"], inplace=True)

df_main_out.to_csv(MAIN_OUT_CSV2, index=False, encoding="utf-8-sig")

print(f"[OK] Added columns: gmd_present (aggregated bool), RQ2 (bool) -> {MAIN_OUT_CSV2}")
print(f"[OK] Wrote {len(out_urls)} rows to: {OUT_URLS}")
try:
    print(out_urls.head(10).to_string(index=False))
except Exception:
    pass


[OK] Added columns: gmd_present (aggregated bool), RQ2 (bool) -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\5.0_Total_Repo.csv
[OK] Wrote 299 rows to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\URL_List_RQ2.csv
                                              repo_url                           full_name
              https://github.com/connectbot/connectbot               connectbot.connectbot
            https://github.com/robolectric/robolectric             robolectric.robolectric
https://github.com/opendocument-app/OpenDocument.droid opendocument-app.opendocument.droid
                https://github.com/maxpower47/PinDroid                 maxpower47.pindroid
                  https://github.com/Rajawali/Rajawali                   rajawali.rajawali
                          https://github.com/cgeo/cgeo                           cgeo.cgeo
      https://github.com/OneBusAway/onebusaway-android       onebusawa

C:\Users\gilla\AppData\Local\Temp\ipykernel_31572\675614616.py:132: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_main["gmd_present"] = df_main["__full__"].map(gmd_any_by_full).fillna(False).astype(bool)
